In [ ]:
from typing import List, Tuple
import os
import re
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    Trainer,
    TrainingArguments,
    DataCollatorForTokenClassification,
    set_seed,
)


os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


MODEL_NAME = "DeepPavlov/rubert-base-cased-conversational"
MODEL_SAVE_DIR = "./fine_tuned_rubert_ner"
TRAIN_PATH = "./train_augmented_simple.csv"
EVAL_PATH = "./submission.csv"


d:\Games\workspace2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [2]:
def build_label2id():
    labels = ["O", "TYPE", "BRAND", "VOLUME", "PERCENT"]
    return {label: i for i, label in enumerate(labels)}

def map_spans_to_tokens(text: str, spans: List[Tuple[int, int, str]], tokenizer) -> List[str]:
    encoded = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    offsets = encoded["offset_mapping"]
    labels = ["O"] * len(offsets)
    for t_idx, (t_start, t_end) in enumerate(offsets):
        for s_start, s_end, s_label in spans:
            if t_start >= s_start and t_end <= s_end:
                labels[t_idx] = s_label
                break
    return labels


In [3]:
def make_hf_dataset(
    texts: List[str],
    annotations: List[List[Tuple[int, int, str]]],
    tokenizer
) -> Dataset:
    label_map = build_label2id()
    packed = {"input_ids": [], "attention_mask": [], "labels": []}

    for text, spans in zip(texts, annotations):
        tok = tokenizer(text, truncation=True, padding=False, max_length=256)
        token_labels = map_spans_to_tokens(text, spans, tokenizer)
        label_ids = [label_map[lbl] for lbl in token_labels]
        label_ids = [-100] + label_ids + [-100]

        packed["input_ids"].append(tok["input_ids"])
        packed["attention_mask"].append(tok["attention_mask"])
        packed["labels"].append(label_ids)

    return Dataset.from_dict(packed)


In [ ]:
def infer_char_spans(text: str, model, tokenizer) -> List[Tuple[int, int, str]]:
    encoded = tokenizer(text, return_tensors="pt", return_offsets_mapping=True)
    offset_map = encoded.pop("offset_mapping")[0].tolist()
    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        pred_ids = outputs.logits.argmax(-1)[0].detach().cpu().numpy()

    id2label = {v: k for k, v in build_label2id().items()}

    spans = []
    current_label = None
    start_pos = None
    prev_end = None


    for p_id, (start, end) in zip(pred_ids[1:-1], offset_map[1:-1]):
        label = id2label[p_id]
        if label != "O":
            if current_label != label:
                if current_label is not None:
                    spans.append((start_pos, prev_end, current_label))
                start_pos = start
                current_label = label
            prev_end = end
        else:
            if current_label is not None:
                spans.append((start_pos, prev_end, current_label))
            current_label = None

    if current_label is not None:
        spans.append((start_pos, prev_end, current_label))

    return spans


def normalize_bio_spans(text: str, raw_spans: List[Tuple[int, int, str]]) -> List[Tuple[int, int, str]]:
    word_bounds = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    if not word_bounds and text.strip():
        word_bounds = [(0, len(text))]

    result = []
    seen = set()
    for w_start, w_end in word_bounds:
        label = "O"
        for s_start, s_end, s_cls in raw_spans:
            if s_start <= w_start and w_end <= s_end:
                label = s_cls
                break
        if label != "O":
            tag = ("I-" if label in seen else "B-") + label
            seen.add(label)
            result.append((w_start, w_end, tag))
    return result

def refine_spans(spans: List[Tuple[int, int, str]], text: str) -> List[Tuple[int, int, str]]:
    refined = []
    for s, e, lbl in spans:
        chunk = text[s:e]
        if lbl.endswith("VOLUME"):
            if re.search(r"\d+(\.\d+)?\s?(мл|л|g|гр|кг)", chunk, flags=re.I):
                refined.append((s, e, lbl))
        elif lbl.endswith("PERCENT"):
            if re.search(r"\d+(\.\d+)?\s?%", chunk):
                refined.append((s, e, lbl))
        else:
            refined.append((s, e, lbl))
    return refined

def predict_spans(text: str, model, tokenizer) -> List[Tuple[int, int, str]]:
    spans = infer_char_spans(text, model, tokenizer)
    spans = normalize_bio_spans(text, spans)
    spans = refine_spans(spans, text)
    return spans


In [5]:
def _strip_bio_suffix(items: List[Tuple[int, int, str]]) -> List[Tuple[int, int, str]]:
    return [(a, b, c.split("-")[-1]) for (a, b, c) in items]

df = pd.read_csv(TRAIN_PATH, delimiter=";")
df["annotation"] = df["annotation"].apply(eval).apply(_strip_bio_suffix)

texts = df["sample"].tolist()
anns = df["annotation"].tolist()


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = make_hf_dataset(texts, anns, tokenizer)

num_labels = len(build_label2id())
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=num_labels).to(device)

for p in model.parameters():
    p.data = p.data.contiguous()

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    logging_dir="./logs",
    logging_steps=500,
    learning_rate=2.1384463785735464e-05,
    weight_decay=0.04793061338540492,
    save_strategy="epoch",
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete.")

trainer.save_model(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)
print(f"Model saved to {MODEL_SAVE_DIR}")


Some weights of BertForTokenClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased-conversational and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
d:\Games\workspace2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Starting fine-tuning...


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_SAVE_DIR)
mdl = AutoModelForTokenClassification.from_pretrained(MODEL_SAVE_DIR).to(device)

# Ожидается df с колонкой 'sample'; при необходимости загрузить отдельно:
# df_eval = pd.read_csv("test.csv", delimiter=";")
# Здесь пример с уже имеющимся df:
df_eval = pd.read_csv("submission.csv", delimiter=";") if os.path.exists("submission.csv") else pd.DataFrame(columns=["sample"])

with torch.no_grad():
    if "sample" in df_eval.columns:
        df_eval["annotation"] = df_eval["sample"].apply(lambda s: predict_spans(s, mdl, tok))

df_eval.to_csv("submission.csv", index=False, sep=";")
print("Saved to submission.csv")
